In [1]:
import sys

sys.path.append("..")
import numpy as np
import os
import pandas as pd
from pathlib import Path
import json
# Add the parent directory to the system path

from classes.sample.raw_data import LMSFile

In [2]:
import scipy.signal as sig
import plotly.express as px
import plotly.graph_objects as go


In [3]:
# path_to_raw = Path("../data/time_examples/2022_11_16_19_44.json")
path_to_raw = Path("../data/time_examples/2022_11_19_08_34_RAW.txt")

In [4]:
lms_file = LMSFile.from_txt(path_to_raw)

In [5]:
x = lms_file.c4_x
dt = lms_file.dt

In [6]:
f, t, s = sig.stft(x, fs=1/dt, nperseg=512, noverlap=400)

In [7]:
fig = go.Figure(
    data=go.Heatmap(
        # z=20*np.log10(np.abs(s)),
        z=np.abs(s),
        x=t,
        y=f,
        colorscale='jet',
        colorbar=dict(title='Amplitude')
    )
)
fig.update_layout(
    title="STFT Spectrogram",
    xaxis_title="Time [s]",
    yaxis_title="Frequency [Hz]"
)
fig.show()


In [8]:
px.line(x=np.arange(len(x)) * dt, y=x)

In [17]:
def compute_cepstrum(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the real cepstrum of each channel (x, y, z) in a vibration signal DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Time-indexed signal with columns ['x', 'y', 'z'].
    fs : float
        Sampling frequency of the signal in Hz.

    Returns
    -------
    pd.DataFrame
        Cepstrum of the signal with columns ['x', 'y', 'z'],
        indexed by quefrency (in seconds).
    """
    dt = df.index.to_series().diff().mean()
    fs = 1 / dt
    # Ensure valid input
    if not {'x', 'y', 'z'}.issubset(df.columns):
        raise ValueError("Input DataFrame must contain columns ['x', 'y', 'z'].")

    N = len(df)
    # Compute FFT for each channel
    fft_vals = np.fft.fft(df[['x', 'y', 'z']], axis=0)

    # Compute log magnitude spectrum
    log_mag = np.log(np.abs(fft_vals) + np.finfo(float).eps)

    # Compute real cepstrum (IFFT of log magnitude)
    cepstrum = np.fft.ifft(log_mag, axis=0).real

    # Define quefrency axis (time-like domain, seconds)
    quefrency = np.arange(N) / fs

    # Return as DataFrame
    cep_df = pd.DataFrame(cepstrum, index=quefrency, columns=['x', 'y', 'z'])
    cep_df.index.name = 'quefrency'

    return cep_df

In [10]:
cable = 3

signal = pd.DataFrame({
    "x": getattr(lms_file, f"c{cable}_x"),
    "y": getattr(lms_file, f"c{cable}_y"),
    "z": getattr(lms_file, f"c{cable}_z"),
})
signal.index = np.arange(len(signal)) * lms_file.dt

In [12]:
px.line(signal)

In [16]:
ceps = compute_cepstrum(signal)